In [1]:
import torch
import numpy as np
import os
from torch.utils.data import DataLoader,Dataset
from torchvision import transforms
from PIL import Image

In [2]:
class ImageProcessor:
    def __init__(self,root_dir_path,transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        self.all_images = [os.path.join(root_dir_path,img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_images)

    def __getitem__(self,idx):
        img_path = self.all_images[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)
        return img

In [3]:
root_dir_path = r"C:\Users\sonu2\anaconda3\anaconda_2025\APNA_COLLEGE\DEEPLEARNING\GAN\img_align_celeba\img_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(175),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [4]:
dataset =ImageProcessor(root_dir_path,transformations)
print(f"loaded: {len(dataset)} images")

loaded: 202599 images


In [5]:
dataloader = DataLoader(dataset,batch_size=128,shuffle=True)

## Generator

In [6]:
import torch.nn as nn
import torch.optim as optim

In [7]:
class Generator(nn.Module):
    def __init__(self,z_dim=100,img_channels=3):
        super(Generator,self).__init__()

        #fully connected
        self.model = nn.Sequential(
            nn.ConvTranspose2d(z_dim,1024,kernel_size=4,stride=1,padding=0),
            nn.BatchNorm2d(1024),
            nn.ReLU(),
    
            nn.ConvTranspose2d(1024,512,kernel_size=4,stride=2,padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
    
            nn.ConvTranspose2d(512,256,kernel_size=4,stride=2,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.ConvTranspose2d(256,128,kernel_size=4,stride=2,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128,img_channels,kernel_size=4,stride=2,padding=1),
            nn.Tanh(),
    )
        
    def forward(self, z):
        z= z.view(z.size(0),z.size(1),1,1) 
        img = self.model(z)
        return img

## Discriminator

In [8]:
class Discriminator(nn.Module):
    def __init__(self,img_channels=3):
        super(Discriminator,self).__init__()

        self.model=nn.Sequential(
            nn.Conv2d(img_channels,128,kernel_size=4,stride=2,padding=1),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Conv2d(128,256,kernel_size=4,stride=2,padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Conv2d(256,512,kernel_size=4,stride=2,padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Conv2d(512,1024,kernel_size=4,stride=2,padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Conv2d(1024,1,kernel_size=4,stride=1,padding=0),
            nn.Sigmoid()
        )

    def forward(self,img):
        out = self.model(img)
        out = out.view(out.size(0),-1)
        return out

In [9]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr= 0.0002, betas=(0.5, 0.999))

In [10]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

cpu


In [11]:
generator = generator.to(device)
discriminator = discriminator.to(device)

## Training 

In [12]:
def train(generator,discriminator,dataloader,epochs=10):
    for epoch in range(epochs):
        for i,imgs in enumerate(dataloader):

            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            real_labels = torch.ones(batch_size,1).to(device)
            fake_labels = torch.zeros(batch_size,1).to(device)

            # TRAIN DISCRIMINATOR
            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size,100,1,1).to(device))

            real_loss = GAN_loss(discriminator(real_imgs),real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()),fake_labels)

            d_loss = (real_loss + fake_loss) / 2

            d_loss.backward()
            d_optimizer.step()

            #TRAIN GENERATOR 
            g_optimizer.zero_grad()

            g_loss = GAN_loss(discriminator(fake_imgs),real_labels)
            g_loss.backward()
            g_optimizer.step()

            if i%50 == 0:
                print(f"for epoch: {epoch+1}/{epochs}... batch:{i+1}...G-Loss:{g_loss.item()}...D-Loss:{d_loss.item()}")

        save_generated_images(generator,epoch,device)
            

In [13]:
import matplotlib.pyplot as plt
import torchvision

In [14]:
def save_generated_images(generator,epoch,device):
    z = torch.randn(num_imgs,100,1,1).to(device)
    generated_images = generator(z).detach().cpu()

    grid = torchvision.utils.make_grid(generated_imgs,n_row=4,normalize=True)

    plt.imshow(np.transpose(grid,(1,2,0)))
    plt.title(f"epoch:{epoch+1}")
    plt.axis("off")
    plt.show()

In [ ]:
train(generator,discriminator,dataloader,epochs=5)

for epoch: 1/5... batch:1...G-Loss:4.35786771774292...D-Loss:0.6924658417701721
